# Voxel — Phase 4: Build a Real Book

Run each cell top to bottom by clicking the ▶ button on the left of each cell.
Wait for one cell to finish (no spinning circle) before running the next.

You'll see the illustrations appear right in this page as they're generated,
so you can catch a bad image immediately instead of waiting for the whole book.

## Step 1 — Get the code from GitHub

In [ ]:
!git clone https://github.com/Wazzaboyzz/Voxel.git
%cd Voxel
!pip install -q -r requirements.txt

## Step 2 — Enter your OpenRouter API key

This box hides what you type. Paste your free OpenRouter key and press Enter.
The key only lives in this Colab session — it is never saved to the repo.

In [ ]:
import getpass, os
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OPENROUTER_API_KEY: ")

## Step 3 — Set the book concept
This is the exact text fed to the AI. Change the line below if you want a different story.

In [ ]:
CONCEPT = "A little bear who is afraid of the dark learns to be brave with the help of his forest friends, bedtime story for ages 3-6"
PAGES = 20
TRIM = "8.5x8.5"
COLORING_BOOK = False   # True = coloring book, False = illustrated story book
PAPER = "white"
print("Concept set:", CONCEPT)

## Step 4 — Generate the manuscript (text for every page)
Read through the pages once they appear. This is your first chance to catch a bad story before spending time on images.

In [ ]:
from content_provider import generate_manuscript

pages = generate_manuscript(CONCEPT, PAGES)
print(f"{len(pages)} pages planned\n")
for p in pages:
    print(f"Page {p['page_number']}: {p['text']}")
    print(f"   image: {p['image_prompt']}\n")

## Step 5 — Generate illustrations (shown inline as they're made)

If any image looks wrong or fails, just re-run THIS cell — it doesn't cost you
the manuscript step above. That's the main advantage of doing this here
instead of a fully automated pipeline.

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage
from image_provider import generate_all_images, BOOK_STYLE_SUFFIX, COLORING_BOOK_STYLE_SUFFIX

safe_name = "".join(c if c.isalnum() or c in " -_" else "" for c in CONCEPT).strip().replace(" ", "_")[:60]
run_dir = Path("output_books") / safe_name
style = COLORING_BOOK_STYLE_SUFFIX if COLORING_BOOK else BOOK_STYLE_SUFFIX

image_files = generate_all_images(
    pages, run_dir / "images", filename_prefix="page",
    width=1600, height=1600, style_suffix=style,
    number_key="page_number", seed_base=100,
)

for f in image_files:
    if f and Path(f).exists():
        display(IPImage(filename=str(f), width=300))
    else:
        print("[missing image]")

## Step 6 — Build the print-ready interior + cover PDFs

In [ ]:
from build_book import build_interior_pdf, build_cover_pdf, TRIM_SIZES
from project_provider import build_project_record, write_project_json

trim_width_in, trim_height_in = TRIM_SIZES[TRIM]

interior_path = run_dir / f"{safe_name}_interior.pdf"
build_interior_pdf(pages, image_files, trim_width_in, trim_height_in, interior_path)
print("Interior PDF:", interior_path)

cover_path = run_dir / f"{safe_name}_cover.pdf"
front_image = image_files[0] if image_files else None
build_cover_pdf(CONCEPT[:40], len(pages), trim_width_in, trim_height_in, cover_path, paper=PAPER, front_image=front_image)
print("Cover PDF:", cover_path)

product_type = "coloring_book" if COLORING_BOOK else "illustrated_book"
record = build_project_record(
    concept=CONCEPT, product_type=product_type,
    trim_width_in=trim_width_in, trim_height_in=trim_height_in,
    paper=PAPER, pages=pages, image_files=image_files,
    interior_path=interior_path, cover_path=cover_path, run_dir=run_dir,
)
project_json_path = write_project_json(record, run_dir)
print("project.json:", project_json_path)

## Step 7 — Download everything to your device
This zips the whole output folder (both PDFs + project.json + images) and
starts a download in your browser.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(safe_name, "zip", run_dir)
files.download(zip_path)

## Next (do this outside Colab, per HANDOFF.md)
1. Open the two PDFs from the download and eyeball them.
2. Run both PDFs through Amazon's **KDP Print Previewer**.
3. Walk KDP's manual listing flow by hand and note friction points.
4. Come back and we scope Phase 5 from what you actually found.